# Compute any ETCCDI extremes index on the Reflective Cloud Hub

**What this is.** One notebook that computes a single climate-extremes index
for the model(s), scenario(s), and ensemble member(s) you choose, and hands
the result back as an xarray object, with an optional quick-look map and an
optional write to the shared S3 layout. It drives the same pipeline
(`multimodel_etccdi.py`) that produced the existing indices dataset, so
anything you compute here is consistent with the released files by
construction: same day-of-year percentile thresholds (SSP2-4.5, 2020-2039,
5-day window), same wet-day masking, same unit conventions.

**Who it is for.** Anyone on the hub who wants one metric without touching the
pipeline internals: pick your choices in the *USER CHOICES* cell, run all
cells, read the result.

**Available indices** (percentile-based; the 16 fixed-threshold indices live
in the companion repo): `R95p`, `R99p` (precipitation), `TX90p`, `TX10p`,
`TN90p`, `TN10p` (temperature exceedances), `WSDI`, `CSDI` (spell lengths).


## Before you run: cost guidance

| Choice | Effect |
|---|---|
| **One model, one scenario** | Fastest. Minutes for a temperature index; precipitation similar. |
| **All four models** | Expect the run to take several times longer; each model loads its own daily data from S3. |
| **More members** | Memory grows roughly linearly with members: each member holds daily fields for the full window in RAM. Three members of daily data for 20 years is the tested configuration. |
| **E3SM** | First-ever access to a variable/period triggers cubed-sphere regridding (ncremap) and caching to the bucket; that first pass is slow (tens of minutes). Later runs stream the cached merged file and are fast. |
| **Monthly frequency (`FREQ='MS'`)** | Same cost as annual (the expensive step is the percentile threshold, shared). Not available for `WSDI`/`CSDI`, whose spells cross month boundaries; the code refuses them at monthly frequency on purpose. |
| **E3SM + temperature indices** | Excluded: E3SM daily max/min temperature are byte-identical at the source, so no true daily extremes exist. The notebook warns and skips. |

**Threshold reuse.** The percentile threshold is computed once per
model-variable and cached in this session; computing a second index on the
same variable (say `TX10p` after `TX90p`... different percentile, so a new
threshold; but `R99p` after `R95p` reuses loaded baseline data) is cheaper
than the first.


In [ ]:
# --- Setup: imports and pipeline path -------------------------------------
import sys, os, warnings
warnings.filterwarnings('ignore')

# Point this at the pipeline src/ if the notebook is not sitting next to it.
SRC = os.path.abspath('src') if os.path.isdir('src') else os.path.abspath('.')
sys.path.insert(0, SRC)

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from multimodel_etccdi import (
    INDEX_REGISTRY, Period,
    compute_baseline_threshold, load_scenario_members,
    compute_index_for_members, ensemble_mean,
    write_index_dataset, ETCCDI_OUTPUT_ROOT,
)
print('Pipeline loaded. Indices available:', ', '.join(INDEX_REGISTRY))


In [ ]:
# --- USER CHOICES: the only cell you need to edit --------------------------

INDEX     = 'TX90p'          # one of: R95p R99p TX90p TX10p TN90p TN10p WSDI CSDI
MODELS    = ['CESM']         # any of: 'CESM', 'UKESM', 'MIROC', 'E3SM'
SCENARIO  = 'HiLLA'          # 'SSP245', 'SAI', or 'HiLLA'
MEMBERS   = None             # None = all available members, or e.g. ['r1'] / ['001']
FREQ      = 'YS'             # 'YS' = annual (default), 'MS' = monthly (not WSDI/CSDI)

ASSESS    = Period('assessment', 2050, 2069)   # the window you want the index over
BASELINE  = Period('baseline', 2020, 2039)     # threshold window (leave as is for
                                               # consistency with the shared dataset)

MAKE_PLOT      = True        # quick-look ensemble-mean map at the end
SAVE_TO_BUCKET = False       # True writes per-member files into the shared layout
                             # (only do this for configurations worth sharing)


In [ ]:
# --- Validation: catch bad combinations before any data loads --------------
assert INDEX in INDEX_REGISTRY, f'{INDEX!r} not in {list(INDEX_REGISTRY)}'
info = INDEX_REGISTRY[INDEX]
VARIABLE, PERCENTILE = info['variable'], info['percentile']

SPELL_INDICES = {'WSDI', 'CSDI'}
if FREQ != 'YS' and INDEX in SPELL_INDICES:
    raise ValueError(f'{INDEX} is spell-based and only defined annually: '
                     'spells cross month boundaries. Use FREQ="YS".')

TEMP_VARS = {'tasmax', 'tasmin'}
models = list(MODELS)
if VARIABLE in TEMP_VARS and 'E3SM' in models:
    print('NOTE: skipping E3SM for a temperature index (no true daily extremes '
          'at the source; see the data-quality note in the pipeline docs).')
    models = [m for m in models if m != 'E3SM']

print(f'Computing {INDEX} ({VARIABLE}, p{PERCENTILE}) | models: {models} | '
      f'{SCENARIO} {ASSESS.start_year}-{ASSESS.end_year} | freq {FREQ}')
if len(models) > 1:
    print('Heads-up: multiple models selected; expect a proportionally longer run.')


In [ ]:
# --- Compute: threshold once per model, then the index per member ----------
results = {}          # model -> {'per_member': {mem: DataArray}, 'ens_mean': DataArray}
thresholds = {}       # model -> threshold DataArray (cached for this session)
baseline_data = {}    # model -> loaded baseline members (cached; reused if you
                      #          compute a second index on the same variable)

for model in models:
    print(f'=== {model} ===')
    if model not in thresholds:
        print('  baseline threshold (percentile_doy over SSP2-4.5 '
              f'{BASELINE.start_year}-{BASELINE.end_year})...')
        # compute_baseline_threshold returns (threshold, baseline_data);
        # keep both so the threshold is a DataArray (not the tuple) and the
        # loaded baseline can be reused.
        thr, base = compute_baseline_threshold(
            model, VARIABLE, PERCENTILE, BASELINE)
        thresholds[model] = thr
        baseline_data[model] = base
    print(f'  loading {SCENARIO} members...')
    data = load_scenario_members(model, VARIABLE, SCENARIO, ASSESS,
                                 members=MEMBERS)
    print(f'  members: {sorted(data)}')
    per_member = compute_index_for_members(data, INDEX, thresholds[model],
                                           ASSESS)
    results[model] = {'per_member': per_member,
                      'ens_mean': ensemble_mean(per_member)}
    print(f'  done: ensemble mean over {len(per_member)} member(s).')

print('All requested computations finished.')

**What you now have.** `results[model]['per_member']` maps each member to
the index averaged over your window (days/year at `FREQ='YS'`), and
`results[model]['ens_mean']` is the member mean, both as `(lat, lon)`
xarray DataArrays ready for your own analysis. Everything below is optional.


In [ ]:
# --- Optional quick-look map ------------------------------------------------
if MAKE_PLOT:
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(6.4 * n, 4.2), squeeze=False)
    for ax, (model, r) in zip(axes[0], results.items()):
        r['ens_mean'].plot(ax=ax, cmap='viridis', cbar_kwargs={'shrink': 0.8})
        ax.set_title(f'{model}: {INDEX}, {SCENARIO} '
                     f'{ASSESS.start_year}-{ASSESS.end_year} (ens. mean)')
    plt.tight_layout()
    plt.show()


In [ ]:
# --- Optional: write per-member annual fields to the shared S3 layout -------
# Uses the pipeline writer, so files land in the same tree as the released
# dataset: {root}/{model}/{scenario}/{member}/{INDEX}.nc
if SAVE_TO_BUCKET:
    for model in models:
        paths = write_index_dataset(model, INDEX, scenarios=(SCENARIO,))
        for p in paths:
            print('wrote', p)
else:
    print('SAVE_TO_BUCKET is False: nothing written. Flip it above to share '
          'this configuration.')


## Extending to a new indicator (ecological, agricultural, ...)

The registry pattern makes new indicators one dictionary entry. An indicator
needs: the daily variable it reads (`pr`, `tasmax`, `tasmin`), the percentile
(or a fixed threshold via the companion fixed-index repo), and the xclim
function that turns daily values plus a threshold into the index. For example,
a growing-degree or frost-day style indicator on `tasmin` follows exactly the
`TN10p` template. Add the entry to `INDEX_REGISTRY` in `multimodel_etccdi.py`,
and this notebook picks it up with no other change: it will appear in the
printed index list and every cell works as before.

If your indicator needs a variable the loaders do not yet carry (e.g. wind or
humidity), that is a loader addition rather than a registry entry: speak to
the pipeline maintainers and we will wire the variable through the per-model
layouts.

## Troubleshooting

- **`FileNotFoundError` on load:** the scenario/member combination is not on
  the archives for that model (e.g. no SAI winds outside CESM; UKESM SSP2-4.5
  sea-level pressure carries two members). The error message names the path
  it looked for.
- **Memory pressure with many members:** run one model at a time; results
  accumulate in the `results` dict across runs of the compute cell if you
  change `MODELS` between runs.
- **E3SM first-run slowness:** that is the one-time regrid+cache; subsequent
  runs stream the merged file from the bucket.
